Notebook for investigating the results at an aggregate level and visualising trends.

## Aggregate Results

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import spatial_hazard as sh
import sim_ranking as sr
import ml_tools as mlt

In [ ]:
# Config
results_dir = Path("/Users/claudy/dev/work/data/sim_ranking/results/gnn/0827_1223")
nzgmdb_ffp = Path("/Users/claudy/dev/work/data/gm_datasets/nz_gmdb/v3.0/Tables/ground_motion_im_table_rotd50_flat.csv")
emp_cim_results_dir = Path("/Users/claudy/dev/work/data/sim_ranking/results/emp_cIM/0826_1336")

In [ ]:
# Load GNN results
gnn_train_results = pd.read_parquet(results_dir / "train_results.parquet")
gnn_val_results = pd.read_parquet(results_dir / "val_results.parquet")
print(f"GNN Train results: {gnn_train_results.shape}")
print(f"GNN Val results: {gnn_val_results.shape}")

gnn_train_events = gnn_train_results.event_id.unique().astype(str)
gnn_val_events = gnn_val_results.event_id.unique().astype(str)

# Load empirical cIM results
train_emp_cim_result = sr.CIMResults.from_dir(emp_cim_results_dir, gnn_train_events)
val_emp_cim_result = sr.CIMResults.from_dir(emp_cim_results_dir, gnn_val_events)

# Load observed data
obs_data = sr.ObservedData.from_nzgmdb_flat(nzgmdb_ffp)

In [ ]:
# Compute GNN residuals
gnn_train_res_df = sr.ml.gnn_gm.get_residuals(gnn_train_results)
gnn_val_res_df = sr.ml.gnn_gm.get_residuals(gnn_val_results)

gnn_train_res_mean_std_df = pd.concat((gnn_train_res_df[sr.constants.PSA_KEYS].mean(axis=0), gnn_train_res_df[sr.constants.PSA_KEYS].std(axis=0)), axis=1)
gnn_train_res_mean_std_df.columns = ["mean", "std"]

gnn_val_res_mean_std_df = pd.concat((gnn_val_res_df[sr.constants.PSA_KEYS].mean(axis=0), gnn_val_res_df[sr.constants.PSA_KEYS].std(axis=0)), axis=1)
gnn_val_res_mean_std_df.columns = ["mean", "std"]

In [ ]:
# Compute empirical cIM residuals
train_emp_cim_res_df = train_emp_cim_result.get_residual_df(obs_data)
val_emp_cim_res_df = val_emp_cim_result.get_residual_df(obs_data)
print(f"Train Emp cIM result: {train_emp_cim_res_df.shape}")
print(f"Val Emp cIM result: {val_emp_cim_res_df.shape}")

In [ ]:
# Plot config
plot_ims = ["pSA_0.1", "pSA_0.5", "pSA_1.0", "pSA_3.0", "pSA_10.0"]

### GNN Residual Visualisation

In [ ]:
# cur_im = "pSA_0.1"
n_bins = 50
bins = np.linspace(-2, 2, n_bins)
for cur_im in plot_ims:
	fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
	
	cur_train_mean = gnn_train_res_mean_std_df.loc[cur_im, "mean"]
	cur_train_std = gnn_train_res_mean_std_df.loc[cur_im, "std"]
	ax1.axvline(cur_train_mean, color="k", linestyle="-", label="Mean", linewidth=1.5)
	ax1.axvline(cur_train_mean + cur_train_std, color="k", linestyle="--", label="Std", linewidth=1.0)
	ax1.axvline(cur_train_mean - cur_train_std, color="k", linestyle="--", linewidth=1.0)
	ax1.hist(gnn_train_res_df[cur_im], bins=bins, label="Train Data")
	ax1.set_xlabel(f"Residual")
	ax1.set_ylabel(f"Count")
	ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
	ax1.set_xlim(-2, 2)
	ax1.set_title("Training Data")
	
	ax1.text(0.02, 0.98, f"$\mu = {cur_train_mean:.2f}$, $\sigma = {cur_train_std:.2}$", horizontalalignment="left", verticalalignment="top", transform=ax1.transAxes)
	
	
	cur_val_mean = gnn_val_res_mean_std_df.loc[cur_im, "mean"]
	cur_val_std = gnn_val_res_mean_std_df.loc[cur_im, "std"]
	ax2.axvline(cur_val_mean, color="k", linestyle="-", label="Mean", linewidth=1.5)
	ax2.axvline(cur_val_mean + cur_val_std, color="k", linestyle="--", label="Std", linewidth=1.0)
	ax2.axvline(cur_val_mean - cur_val_std, color="k", linestyle="--", linewidth=1.0)
	ax2.hist(gnn_val_res_df[cur_im], bins=bins, label="Val Data")
	ax2.set_xlabel(f"Residual")
	ax2.set_ylabel(f"Count")
	ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
	ax2.set_xlim(-2, 2)
	ax2.set_title("Validation Data")
	
	ax2.text(0.02, 0.98, f"$\mu = {cur_val_mean:.2f}$, $\sigma = {cur_val_std:.2}$", horizontalalignment="left", verticalalignment="top", transform=ax2.transAxes)
	
	fig.suptitle(f"{cur_im}");
	fig.tight_layout()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.semilogx(sr.constants.PERIODS, gnn_train_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], label="Training", c="b")
ax1.semilogx(sr.constants.PERIODS, gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], label="Validation", c="b", linestyle="--")
ax1.axhline(0, color="k", linestyle="--")
ax1.set_xlabel(f"Period (s)")
ax1.set_ylabel(f"Bias")
ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax1.set_xlim(0.01, 10)
ax1.set_ylim(-0.5, 0.5)

ax2.semilogx(sr.constants.PERIODS, gnn_train_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], label="Training", c="b")
ax2.semilogx(sr.constants.PERIODS, gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], label="Validation", c="b", linestyle="--")
ax2.set_xlabel(f"Period (s)")
ax2.set_ylabel(f"Standard deviation of residuals")
ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax2.set_xlim(0.01, 10);
ax2.set_ylim(0, 1)
ax2.legend()


### Comparison with empirical cIM approach

In [ ]:
# Get common events
train_events = np.intersect1d(gnn_train_events, train_emp_cim_result.events)
val_events = np.intersect1d(gnn_val_events, val_emp_cim_result.events)

common_train_scenarios = np.intersect1d(gnn_train_res_df.index.values.astype(str), train_emp_cim_res_df.index.values.astype(str))
common_val_scenarios = np.intersect1d(gnn_val_res_df.index.values.astype(str), val_emp_cim_res_df.index.values.astype(str))
print(f"Common train scenarios: {len(common_train_scenarios)}")
print(f"Common val scenarios: {len(common_val_scenarios)}")

cmp_gnn_train_res_df = gnn_train_res_df.loc[common_train_scenarios]
cmp_gnn_val_res_df = gnn_val_res_df.loc[common_val_scenarios]

cmp_gnn_train_res_mean_std_df = pd.concat((cmp_gnn_train_res_df[sr.constants.PSA_KEYS].mean(axis=0), cmp_gnn_train_res_df[sr.constants.PSA_KEYS].std(axis=0)), axis=1)
cmp_gnn_train_res_mean_std_df.columns = ["mean", "std"]

cmp_gnn_val_res_mean_std_df = pd.concat((cmp_gnn_val_res_df[sr.constants.PSA_KEYS].mean(axis=0), cmp_gnn_val_res_df[sr.constants.PSA_KEYS].std(axis=0)), axis=1)
cmp_gnn_val_res_mean_std_df.columns = ["mean", "std"]

cmp_emp_cim_train_res_df = train_emp_cim_res_df.loc[common_train_scenarios]
cmp_emp_cim_val_res_df = val_emp_cim_res_df.loc[common_val_scenarios]

cmp_emp_cim_train_res_mean_std_df = pd.concat((cmp_emp_cim_train_res_df[sr.constants.PSA_KEYS].mean(axis=0), cmp_emp_cim_train_res_df[sr.constants.PSA_KEYS].std(axis=0)), axis=1)
cmp_emp_cim_train_res_mean_std_df.columns = ["mean", "std"]

cmp_emp_cim_val_res_mean_std_df = pd.concat((cmp_emp_cim_val_res_df[sr.constants.PSA_KEYS].mean(axis=0), cmp_emp_cim_val_res_df[sr.constants.PSA_KEYS].std(axis=0)), axis=1)
cmp_emp_cim_val_res_mean_std_df.columns = ["mean", "std"]

In [ ]:
# cur_im = "pSA_0.1"
n_bins = 50
bins = np.linspace(-2, 2, n_bins)
for cur_im in plot_ims:
	fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
	
	cur_mean = cmp_emp_cim_val_res_df[cur_im].mean(axis=0)
	cur_std =  cmp_emp_cim_val_res_df[cur_im].std(axis=0)
	ax1.axvline(cur_mean, color="k", linestyle="-", label="Mean", linewidth=1.5)
	ax1.axvline(cur_mean + cur_std, color="k", linestyle="--", label="Std", linewidth=1.0)
	ax1.axvline(cur_mean - cur_std, color="k", linestyle="--", linewidth=1.0)
	ax1.hist(cmp_emp_cim_val_res_df[cur_im], bins=bins, label="Train Data")
	ax1.set_xlabel(f"Residual")
	ax1.set_ylabel(f"Count")
	ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
	ax1.set_xlim(-2, 2)
	ax1.set_title("Empirical cIM")
	
	ax1.text(0.02, 0.98, f"$\mu = {cur_mean:.2f}$, $\sigma = {cur_std:.2}$", horizontalalignment="left", verticalalignment="top", transform=ax1.transAxes)
	
	
	cur_mean = cmp_gnn_val_res_df[cur_im].mean(axis=0)
	cur_std =  cmp_gnn_val_res_df[cur_im].std(axis=0)
	ax2.axvline(cur_mean, color="k", linestyle="-", label="Mean", linewidth=1.5)
	ax2.axvline(cur_mean + cur_std, color="k", linestyle="--", label="Std", linewidth=1.0)
	ax2.axvline(cur_mean - cur_std, color="k", linestyle="--", linewidth=1.0)
	ax2.hist(cmp_gnn_val_res_df[cur_im], bins=bins, label="Val Data")
	ax2.set_xlabel(f"Residual")
	ax2.set_ylabel(f"Count")
	ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
	ax2.set_xlim(-2, 2)
	ax2.set_title("ML")
	
	ax2.text(0.02, 0.98, f"$\mu = {cur_mean:.2f}$, $\sigma = {cur_std:.2}$", horizontalalignment="left", verticalalignment="top", transform=ax2.transAxes)
	
	fig.suptitle(f"{cur_im}");
	fig.tight_layout()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.semilogx(sr.constants.PERIODS, cmp_gnn_train_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], label="ML - Training", c="b")
ax1.semilogx(sr.constants.PERIODS, cmp_gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], label="ML - Validation", c="b", linestyle="--")
ax1.semilogx(sr.constants.PERIODS, cmp_emp_cim_train_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], label="Empirical cIM - Training", c="g")
ax1.semilogx(sr.constants.PERIODS, cmp_emp_cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], label="Empirical cIM - Validation", linestyle="--", c="g")
ax1.axhline(0, color="k", linestyle="--")
ax1.set_xlabel(f"Period (s)")
ax1.set_ylabel(f"Bias")
ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax1.set_xlim(0.01, 10)
ax1.set_ylim(-0.5, 0.5)

ax2.semilogx(sr.constants.PERIODS, cmp_gnn_train_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], label="ML - Training", c="b")
ax2.semilogx(sr.constants.PERIODS, cmp_gnn_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], label="ML - Validation", c="b", linestyle="--")
ax2.semilogx(sr.constants.PERIODS, cmp_emp_cim_train_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], label="Empirical cIM - Training", c="g")
ax2.semilogx(sr.constants.PERIODS, cmp_emp_cim_val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], label="Empirical cIM - Validation", linestyle="--", c="g")
ax2.set_xlabel(f"Period (s)")
ax2.set_ylabel(f"Standard deviation of residuals")
ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax2.set_xlim(0.01, 10)
ax2.legend();

## Trends
Look at some trends of these results

### GNN - Magnitude


In [ ]:
for cur_im in plot_ims:
	fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
	
	cur_x = obs_data.event_df.loc[gnn_train_results.event_id, "mag"]
	cur_y = gnn_train_res_df[cur_im]
	cur_bin_centres, cur_bin_means, cur_bin_stds = mlt.utils.compute_count_binned_trend(cur_x.values, cur_y.values, n_points_per_bin=250)
	ax1.scatter(cur_x, cur_y, label="Training", c="b", alpha=0.25, zorder=0)
	ax1.plot(cur_bin_centres, cur_bin_means, c="k", label="Mean", )
	ax1.plot(cur_bin_centres, cur_bin_means + cur_bin_stds, c="k", linestyle="--", label="Std")
	ax1.plot(cur_bin_centres, cur_bin_means - cur_bin_stds, c="k", linestyle="--")
	
	ax1.set_xlabel(f"Magnitude")
	ax1.set_ylabel(f"Residual")
	ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
	ax1.set_xlim(3, 7.5)
	ax1.set_ylim(-2.5, 2.5)
	
	
	cur_x = obs_data.event_df.loc[gnn_val_results.event_id, "mag"]
	cur_y = gnn_val_res_df[cur_im]
	cur_bin_centres, cur_bin_means, cur_bin_stds = mlt.utils.compute_count_binned_trend(cur_x.values, cur_y.values, n_points_per_bin=250)
	ax2.scatter(cur_x, cur_y, label="Validation", c="b", alpha=0.25)
	ax2.plot(cur_bin_centres, cur_bin_means, c="k", label="Mean")
	ax2.plot(cur_bin_centres, cur_bin_means + cur_bin_stds, c="k", linestyle="--", label="Std")
	ax2.plot(cur_bin_centres, cur_bin_means - cur_bin_stds, c="k", linestyle="--")
	ax2.set_xlabel(f"Magnitude")
	ax2.set_ylabel(f"Residual")
	ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
	ax2.set_xlim(3, 7.5)
	ax2.set_ylim(-2.5, 2.5)
	
	fig.suptitle(f"{cur_im}");
	fig.tight_layout()
	


### GNN - Rrup

In [ ]:
for cur_im in plot_ims:
	fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
	
	cur_x = obs_data.record_df.loc[gnn_train_results.index, "rrup"]
	cur_y = gnn_train_res_df[cur_im]
	cur_bin_centres, cur_bin_means, cur_bin_stds = mlt.utils.compute_count_binned_trend(cur_x.values, cur_y.values, n_points_per_bin=250)
	ax1.scatter(cur_x, cur_y, label="Training", c="b", alpha=0.25, zorder=0)
	ax1.plot(cur_bin_centres, cur_bin_means, c="k", label="Mean", )
	ax1.plot(cur_bin_centres, cur_bin_means + cur_bin_stds, c="k", linestyle="--", label="Std")
	ax1.plot(cur_bin_centres, cur_bin_means - cur_bin_stds, c="k", linestyle="--")
	
	ax1.set_xlabel(f"Rrup")
	ax1.set_ylabel(f"Residual")
	ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
	ax1.set_xlim(2, 200)
	ax1.set_ylim(-2.5, 2.5)
	ax1.set_xscale("log")
	
	
	cur_x = obs_data.record_df.loc[gnn_val_results.index, "rrup"]
	cur_y = gnn_val_res_df[cur_im]
	cur_bin_centres, cur_bin_means, cur_bin_stds = mlt.utils.compute_count_binned_trend(cur_x.values, cur_y.values, n_points_per_bin=250)
	ax2.scatter(cur_x, cur_y, label="Validation", c="b", alpha=0.25)
	ax2.plot(cur_bin_centres, cur_bin_means, c="k", label="Mean")
	ax2.plot(cur_bin_centres, cur_bin_means + cur_bin_stds, c="k", linestyle="--", label="Std")
	ax2.plot(cur_bin_centres, cur_bin_means - cur_bin_stds, c="k", linestyle="--")
	ax2.set_xlabel(f"Rrup")
	ax2.set_ylabel(f"Residual")
	ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
	ax2.set_xlim(2, 200)

### GNN - Constraintness

In [ ]:
# Get the Loth and Baker correlation data
dist_matrix = sh.im_dist.calculate_distance_matrix(obs_data.sites, obs_data.site_df)
lb_corr_data = sr.LBSiteCorrelationData.from_dist_matrix(dist_matrix, sr.constants.PSA_KEYS) 

In [ ]:
# Compute the constraintness for each scenario
gnn_train_constraint = {}
for cur_ix, cur_row in gnn_train_results.iterrows():
	cur_corr_data = lb_corr_data.corr_data.sel[cur_row.site_int,  :, :].loc[cur_row.obs_sites]
	cur_constraint = cur_corr_data.sum(axis=0).mean()
	gnn_train_constraint[cur_ix] = cur_constraint

gnn_train_constraint = pd.Series(gnn_train_constraint, name="constraint")

gnn_val_constraint = {}
for cur_ix, cur_row in gnn_val_results.iterrows():
	cur_corr_data = lb_corr_data.corr_data.sel[cur_row.site_int,  :, :].loc[cur_row.obs_sites]
	cur_constraint = cur_corr_data.sum(axis=0).mean()
	gnn_val_constraint[cur_ix] = cur_constraint
    
gnn_val_constraint = pd.Series(gnn_val_constraint, name="constraint")

#### Distribution of constraintness

In [ ]:
cur_x_min = min(gnn_train_constraint.min(), gnn_val_constraint.min())
cur_x_max = max(gnn_train_constraint.max(), gnn_val_constraint.max())

for cur_im in plot_ims:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    ax1.hist(gnn_train_constraint, bins=50, label="Training", alpha=0.5)
    ax1.set_xlabel(f"Constraintness")
    ax1.set_ylabel(f"Count")
    ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    ax1.set_title("Training Data")
    ax1.set_xlim(cur_x_min, cur_x_max)
    
    ax2.hist(gnn_val_constraint, bins=50, label="Validation", alpha=0.5)
    ax2.set_xlabel(f"Constraintness")
    ax2.set_ylabel(f"Count")
    ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    ax2.set_title("Validation Data")
    ax2.set_xlim(cur_x_min, cur_x_max)
    
    fig.suptitle(f"{cur_im}");
    fig.tight_layout()

### GNN Residuals as a function of constraintness

In [ ]:
cur_x_min = min(gnn_train_constraint.min(), gnn_val_constraint.min())
cur_x_max = max(gnn_train_constraint.max(), gnn_val_constraint.max())
	
for cur_im in plot_ims:	
	fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
	
	cur_x = gnn_train_constraint
	cur_y = gnn_train_res_df[cur_im]
	cur_bin_centres, cur_bin_means, cur_bin_stds = mlt.utils.compute_count_binned_trend(cur_x.values, cur_y.values, n_points_per_bin=250)
	ax1.scatter(cur_x, cur_y, label="Training", c="b", alpha=0.25, zorder=0, s=10)
	ax1.plot(cur_bin_centres, cur_bin_means, c="k", label="Mean", )
	ax1.plot(cur_bin_centres, cur_bin_means + cur_bin_stds, c="k", linestyle="--", label="Std")
	ax1.plot(cur_bin_centres, cur_bin_means - cur_bin_stds, c="k", linestyle="--")
	ax1.set_xlabel(f"Constraintness")
	ax1.set_ylabel(f"Residual")
	ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
	ax1.set_xlim(cur_x_min, cur_x_max)
	ax1.set_ylim(-2.5, 2.5)
	ax1.set_xscale("log")
	ax1.set_title("Training Data")
	
	cur_x = gnn_val_constraint
	cur_y = gnn_val_res_df[cur_im]
	cur_bin_centres, cur_bin_means, cur_bin_stds = mlt.utils.compute_count_binned_trend(cur_x.values, cur_y.values, n_points_per_bin=250)
	ax2.scatter(cur_x, cur_y, label="Validation", c="b", alpha=0.25, s=10)
	ax2.plot(cur_bin_centres, cur_bin_means, c="k", label="Mean")
	ax2.plot(cur_bin_centres, cur_bin_means + cur_bin_stds, c="k", linestyle="--", label="Std")
	ax2.plot(cur_bin_centres, cur_bin_means - cur_bin_stds, c="k", linestyle="--")
	ax2.set_xlabel(f"Constraintness")
	ax2.set_ylabel(f"Residual")
	ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
	ax2.set_xlim(cur_x_min, cur_x_max)
	ax2.set_ylim(-2.5, 2.5)
	ax2.set_xscale("log")
	ax2.set_title("Validation Data")
	
	fig.suptitle(f"{cur_im}");
	fig.tight_layout()
	


### Predicted Standard deviation as a function of constraintness

In [ ]:
cur_x_min = min(gnn_train_constraint.min(), gnn_val_constraint.min())
cur_x_max = max(gnn_train_constraint.max(), gnn_val_constraint.max())

for cur_im in plot_ims:
	fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
	
	cur_x = gnn_train_constraint
	cur_y = gnn_train_results[f"{cur_im}_pred_std"]
	cur_bin_centres, cur_bin_means, cur_bin_stds = mlt.utils.compute_count_binned_trend(cur_x.values, cur_y.values, n_points_per_bin=250)
	ax1.scatter(cur_x, cur_y, label="Training", alpha=0.25, s=10)
	ax1.plot(cur_bin_centres, cur_bin_means, c="k", label="Mean", )
	ax1.plot(cur_bin_centres, cur_bin_means + cur_bin_stds, c="k", linestyle="--", label="Std")
	ax1.plot(cur_bin_centres, cur_bin_means - cur_bin_stds, c="k", linestyle="--")
	ax1.set_xlabel(f"Constraintness")
	ax1.set_ylabel(f"Predicted Standard Deviation")
	ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
	ax1.set_xlim(cur_x_min, cur_x_max)
	ax1.set_ylim(0.2, 1.5)
	ax1.set_xscale("log")
	ax1.set_title("Training Data")
	
	cur_x = gnn_val_constraint
	cur_y = gnn_val_results[f"{cur_im}_pred_std"]
	cur_bin_centres, cur_bin_means, cur_bin_stds = mlt.utils.compute_count_binned_trend(cur_x.values, cur_y.values, n_points_per_bin=250)
	ax2.scatter(cur_x, cur_y, label="Validation", alpha=0.25, s=10)
	ax2.plot(cur_bin_centres, cur_bin_means, c="k", label="Mean")
	ax2.plot(cur_bin_centres, cur_bin_means + cur_bin_stds, c="k", linestyle="--", label="Std")
	ax2.plot(cur_bin_centres, cur_bin_means - cur_bin_stds, c="k", linestyle="--")
	ax2.set_xlabel(f"Constraintness")
	ax2.set_ylabel(f"Predicted Standard Deviation")
	ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
	ax2.set_xlim(cur_x_min, cur_x_max)
	ax2.set_ylim(0.2, 1.5)
	ax2.set_xscale("log")
	ax2.set_title("Validation Data")
	
	fig.suptitle(f"{cur_im}");
	fig.tight_layout()

### Comparison with empirical cIM